# CSC2107: Design & Analysis of Algorithms
## Assignment 5.1 — Naive vs. Memoized Fibonacci


**Objective:** Implement plain recursive Fibonacci and recursive Fibonacci with memoization, compare their function-call counts and execution times, plot the results, and analyze their complexity.



## Part 1 — Implementation

The Fibonacci sequence is defined by:

- \(F(0)=0\)
- \(F(1)=1\)
- \(F(n)=F(n-1)+F(n-2)\), for \(n\geq2\)

Algorithm A uses plain recursion. Algorithm B uses a dictionary as a memoization table to reuse values already calculated.


In [1]:
import time
import sys
import csv
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("Python recursion limit:", sys.getrecursionlimit())


Python recursion limit: 1000


### Algorithm A: Plain recursion

The global counter `calls_a` increases every time the function is entered, including calls that return the base cases.


In [ ]:
calls_a = 0

def fib_naive(n):
    global calls_a
    calls_a += 1

    if n == 0 or n == 1:
        return n

    return fib_naive(n - 1) + fib_naive(n - 2)


In [ ]:
plt.figure(figsize=(9, 5.5))

plt.plot(
    naive_df["n"],
    naive_df["calls"],
    marker="o",
    label="Algorithm A: Naive"
)

plt.yscale("log")
plt.xlabel("n")
plt.ylabel("Number of function calls (log scale)")
plt.title("Algorithm A: Naive Fibonacci")
plt.grid(True, which="both")
plt.legend()
plt.show()

### Algorithm B: Recursion with memoization

The dictionary `fibtable` stores computed Fibonacci values. The test `n in fibtable` checks whether the key exists, even if the stored value is zero.


In [ ]:
calls_b = 0
fibtable = {}

def fib_memo(n):
    global calls_b
    calls_b += 1

    if n in fibtable:
        return fibtable[n]

    if n == 0 or n == 1:
        value = n
    else:
        value = fib_memo(n - 1) + fib_memo(n - 2)

    fibtable[n] = value
    return value


### Check correctness for n = 0 to 20

The assertion stops the cell if the two algorithms return different results.


In [ ]:
correctness_results = []

for n in range(21):
    calls_a = 0
    result_a = fib_naive(n)

    calls_b = 0
    fibtable = {}
    result_b = fib_memo(n)

    assert result_a == result_b, f"Mismatch at n={n}"
    correctness_results.append({
        "n": n,
        "F(n)": result_a,
        "Naive calls": calls_a,
        "Memoized calls": calls_b
    })

display(pd.DataFrame(correctness_results))
print("Correctness test passed: both algorithms agree for n = 0 to 20.")


## Part 2 — Experiments

For each run, measure elapsed time with `time.perf_counter()` and count function calls. The memoization table is cleared before every memoized run. The naive experiment stops if a run takes longer than 60 seconds.


### Algorithm A: Naive recursion

Requested inputs: 5, 10, 15, 20, 25, 30, and 35. If a run exceeds 60 seconds, it is recorded and the experiment stops.


In [ ]:
naive_rows = []

for n in [5, 10, 15, 20, 25, 30, 35]:
    calls_a = 0
    start = time.perf_counter()
    value = fib_naive(n)
    elapsed = time.perf_counter() - start

    naive_rows.append({
        "algorithm": "Naive",
        "n": n,
        "calls": calls_a,
        "time (s)": elapsed,
        "F(n)": value
    })

    print(f"n={n:>2} | calls={calls_a:>12,} | time={elapsed:.6f} s")

    if elapsed > 60:
        print("A run exceeded 60 seconds. Stopping the naive experiment.")
        break

naive_df = pd.DataFrame(naive_rows)
display(naive_df)


### Algorithm B: Memoized recursion

Requested inputs: 10, 50, 100, 200, 500, and 1000. A `RecursionError` is recorded if the current Python recursion limit prevents a run from completing.


In [ ]:
memo_rows = []

for n in [10, 50, 100, 200, 500, 1000]:
    calls_b = 0
    fibtable = {}  # Important: clear the cache before each run

    start = time.perf_counter()
    try:
        value = fib_memo(n)
        status = "Completed"
    except RecursionError:
        value = None
        status = "RecursionError"
    elapsed = time.perf_counter() - start

    memo_rows.append({
        "algorithm": "Memoized",
        "n": n,
        "calls": calls_b,
        "time (s)": elapsed,
        "F(n)": value,
        "status": status
    })

    print(f"n={n:>4} | calls={calls_b:>6,} | time={elapsed:.6f} s | {status}")

memo_df = pd.DataFrame(memo_rows)
display(memo_df)


### Combined results





In [ ]:
all_results = pd.concat([naive_df, memo_df], ignore_index=True, sort=False)
display(all_results)

all_results.to_csv("fibonacci_results.csv", index=False)
print("Saved fibonacci_results.csv")


### Graph: n against number of function calls

A logarithmic y-axis is used because the naive algorithm's call count grows exponentially. This makes it possible to view the much smaller memoized call counts on the same graph.


In [ ]:
plt.figure(figsize=(9, 5.5))

plt.plot(naive_df["n"], naive_df["calls"], marker="o", label="Naive recursion")
plt.plot(memo_df["n"], memo_df["calls"], marker="s", label="Memoized recursion")

plt.yscale("log")
plt.xlabel("n (Fibonacci index)")
plt.ylabel("Number of function calls (log scale)")
plt.title("Naive vs. Memoized Fibonacci")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("fibonacci_calls.png", dpi=200)
plt.show()

print("Saved fibonacci_calls.png")
